# 01：亲手走一遍 Flow Matching

这个 notebook 不训练大模型，也不下载数据。先把每个张量的输入、操作、输出讲清，再运行命令行完整实验。

先在仓库根目录的虚拟环境执行 `pip install -e .`，用同一个环境启动 Jupyter。只运行下面代码不需要 Jupyter 依赖；自动测试会按顺序执行全部代码单元。

In [ ]:
import torch
from flowlab.data import generator, sample_base, sample_target
from flowlab.path import straight_path
from flowlab.model import VelocityMLP
from flowlab.losses import velocity_mse
from flowlab.solvers import integrate

torch.set_num_threads(1)
torch.manual_seed(42)
print("torch:", torch.__version__, "device: cpu, dtype: float32")


## 1. 一对点，三个不同的对象

- z0：噪声端点；z1：数据端点
- zt：沿这条配对直线、时刻 t 的位置
- u：这条直线的速度，单位是“坐标 / 时间”，不是终点

约定固定：t=0 是噪声，t=1 是数据。对独立配对的样本，zt=(1-t)z0+t*z1，u=z1-z0。

In [ ]:
z0 = torch.tensor([[0.0, 2.0]])
z1 = torch.tensor([[4.0, 0.0]])
t = torch.tensor([[0.25]])
zt, u = straight_path(z0, z1, t)
prediction = torch.tensor([[3.0, -1.0]])
loss = velocity_mse(prediction, u)
print("zt =", zt.tolist(), "u =", u.tolist(), "MSE =", loss.item())
assert torch.equal(zt, torch.tensor([[1.0, 1.5]]))
assert torch.equal(u, torch.tensor([[4.0, -2.0]]))
assert loss.item() == 1.0  # ((3-4)^2 + (-1-(-2))^2) / 2


## 2. 训练输入不是原始点对

网络只看 zt 和 t，不知道本次 z0、z1。输出二维速度。很多配对路径可能经过附近，因此网络学习的是给定 zt,t 的平均速度场；它不需要把每一条配对直线都记下来。

下面的 shape 是 `[batch, coordinates]`。标签只用于计算损失，不作为网络输入。

In [ ]:
rng = generator(12)
batch_z0 = sample_base(8, rng)
batch_z1 = sample_target(8, "moons", rng)
batch_t = torch.rand(8, 1, generator=rng)
batch_zt, target_u = straight_path(batch_z0, batch_z1, batch_t)
model = VelocityMLP(hidden_dim=32, hidden_layers=2)
predicted_u = model(batch_zt, batch_t)
print({"zt": list(batch_zt.shape), "t": list(batch_t.shape),
       "prediction": list(predicted_u.shape), "target": list(target_u.shape)})


## 3. 权重什么时候变化？

前向计算输出速度；反向传播计算梯度；optimizer.step() 更新参数 θ。训练中的位置 zt 是由解析路径直接构造的，**不需要先运行 ODE 求解器**。

下面只做一步训练，作用是观察参数改变，不能据此声称生成效果好。

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
before = {name: p.detach().clone() for name, p in model.named_parameters()}
train_loss = velocity_mse(predicted_u, target_u)
optimizer.zero_grad(set_to_none=True)
train_loss.backward()
optimizer.step()
change = sum((p.detach() - before[name]).abs().sum().item()
             for name, p in model.named_parameters())
print("one-step loss:", train_loss.item(), "sum |weight change|:", change)
assert change > 0


## 4. 推理时变化的是采样状态 z

使用固定 θ，多次调用 vθ(z,t)。Euler 一步是 z_next=z+Δt*vθ(z,t)；midpoint 先估计半步位置，再查询半步速度。

推理不使用本次的目标 z1，也不执行 optimizer.step()。从标准高斯重新抽初始状态才产生新样本。同一 checkpoint、同一初始噪声才能公平比较求解器。

In [ ]:
model.eval()
saved_weights = {k: v.clone() for k, v in model.state_dict().items()}
initial = sample_base(4, generator(99))
euler = integrate(model, initial, steps=8, method="euler")
midpoint = integrate(model, initial, steps=8, method="midpoint")
print("state trajectory shape:", list(midpoint.trajectory.shape))
print("Euler NFE:", euler.nfe, "midpoint NFE:", midpoint.nfe)
assert all(torch.equal(v, saved_weights[k]) for k, v in model.state_dict().items())
assert torch.equal(midpoint.trajectory[0], initial)


## 5. 不依赖学习质量的求解器验算

如果场恒等于 (4,-2)，积分一整个时间单位就应该把 (0,2) 推到 (4,0)。这能检查时间方向、步长和更新公式，与网络是否训练好无关。

In [ ]:
constant_velocity = torch.tensor([4.0, -2.0])
constant_field = lambda z, t: constant_velocity.expand_as(z)
for method in ("euler", "midpoint"):
    result = integrate(constant_field, z0, steps=4, method=method)
    print(method, result.final.tolist())
    torch.testing.assert_close(result.final, z1)


## 6. 进入可复现实验

回到命令行：

```bash
python -m flowlab train --config configs/moons.json --out runs/moons
python -m flowlab compare --checkpoint runs/moons/checkpoint.pt --out runs/compare --steps 4 8 16 32 64
```

按照 `docs/experiment_worksheet.md`，先写预测，再只改一个变量。读 `loss.png` 看拟合，读 `samples.png` 看分布，读 `trajectories.png` 看状态变化，读 `vector_field.png` 看不同时间的网络输出。

训练 target 的直线速度恒定，不意味着学到的边缘速度场恒定，也不意味着实际生成轨迹必须是直线。